# e6 – Overall comparison to the benchmark (`hist_rw`)

Does each model forecast next month's realised variance better than the historical-volatility
benchmark `hist_rw` (the forecast for month t+1 is month t's realised variance)?
Models come from `research_config.SELECTED`; each is compared with `BENCHMARK`.

**Tests:**
Every Elastic Net variant includes `log_rv_1m`, so it nests `hist_rw` (intercept 0, coefficient 1
on `log_rv_1m`, 0 on the rest). For these, log-RV MSE uses Clark-West, with DM for the raw loss
difference. GARCH, GARCH-MIDAS and XGBoost do not nest `hist_rw`, so log-RV MSE uses DM only.
QLIKE uses DM for every model. Pooled differentials are averaged across ETFs each month and tested
with Newey-West (Andrews bandwidth). P-values are one-sided, H1: `a` beats `b`.

Clark-West is approximate for tuned Elastic Net and can be significant despite
worse raw MSE. DM p-values also require caution for nested comparisons.

Output in `forecasting/results/`: `e6_losses.csv`, `e6_evaluation.csv` 
`a` is the model and `b` is `hist_rw`. `gain_t > 0` favours `a`; `loss_ratio` is loss(a) / loss(b).
`pct_change = (loss_ratio - 1) * 100`; negative means lower raw loss for `a`.

In [9]:
import sys
from pathlib import Path

_checkout = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "models" / "research_config.py").is_file()),
    None,
)
if _checkout is None:
    raise FileNotFoundError("Start the notebook kernel from within this project checkout.")
sys.path.insert(0, str(_checkout / "models"))
from utils.paths import PROJECT_ROOT, MODELS_DIR, DATASET_DIR, FORECAST_DIR, FORECAST_RESULTS_DIR

import sys
from importlib import reload
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import norm

from utils.paths import FORECAST_DIR, FORECAST_RESULTS_DIR as RESULTS_DIR
from utils.portfolio_utils import FORECAST_FILES
import research_config

# Notebook kernels cache earlier comparison definitions between runs.
reload(research_config)
from research_config import SELECTED, BENCHMARK, TICKERS

RESULTS_DIR.mkdir(exist_ok=True)

# Families whose variants nest hist_rw: each Elastic Net includes log_rv_1m.
NESTED_FAMILIES = {"Elastic Net"}

frames = []
for fam in SELECTED:
    f = pd.read_csv(FORECAST_DIR / FORECAST_FILES[fam], parse_dates=["date"]).assign(family=fam)
    # Older exports may omit target_date; all forecasts target the next month end.
    next_month = f["date"] + pd.offsets.MonthEnd(1)
    if "target_date" in f and not pd.to_datetime(f["target_date"]).eq(next_month).all():
        raise ValueError(f"{FORECAST_FILES[fam]} targets a month other than the next one.")
    frames.append(f.assign(target_date=next_month))
df = pd.concat(frames, ignore_index=True)

selected = [m for ms in SELECTED.values() for m in ms]
available = set(df["model"])
for m in selected:
    if m not in available:
        print(f"NOTICE: {m} has no forecasts yet; it and its pair are skipped")
models = [m for m in selected if m in available]
if BENCHMARK not in models:
    raise ValueError(f"{BENCHMARK} has no forecasts.")
df = df[df["model"].isin(models) & df["etf"].isin(TICKERS)].copy()
if df.duplicated(["model", "etf", "date"]).any():
    raise ValueError("Duplicate model/ETF/forecast-month rows.")
if df["date"].isna().any():
    raise ValueError("Forecast dates must be present.")

family = {m: fam for fam, ms in SELECTED.items() for m in ms}
nested = {m: family[m] in NESTED_FAMILIES for m in models}
pairs = [(f"E6.{i}", m, BENCHMARK, family[m])
         for i, m in enumerate([m for m in models if m != BENCHMARK], 1)]
print(f"{len(models)} of {len(selected)} models available; {len(pairs)} comparisons with {BENCHMARK}")

11 of 11 models available; 10 comparisons with hist_rw


In [10]:
def andrews_bandwidth(x):
    x = np.asarray(x, float)
    n = len(x)
    xd = x - x.mean()
    den = float(xd[:-1] @ xd[:-1])
    rho = 0.0 if den <= 0 else float(xd[1:] @ xd[:-1]) / den
    rho = min(max(rho, -0.97), 0.97)
    a1 = 4.0 * rho ** 2 / ((1 - rho) ** 2 * (1 + rho) ** 2)
    return 0 if a1 <= 0 else int(min(np.floor(1.1447 * (a1 * n) ** (1 / 3)), n // 4))


def newey_west_lrv(x, lag):
    x = np.asarray(x, float)
    n = len(x)
    xd = x - x.mean()
    lag = max(0, min(lag, n - 1))
    s = float(xd @ xd) / n
    for l in range(1, lag + 1):
        s += 2.0 * (1.0 - l / (lag + 1.0)) * float(xd[l:] @ xd[:-l]) / n
    return max(s, 1e-16)


def hac_t(d):
    """t-statistic of mean(d) with a Newey-West variance."""
    d = np.asarray(d, float)
    return d.mean() / np.sqrt(newey_west_lrv(d, andrews_bandwidth(d)) / len(d))


def qlike(rv, fc):
    r = rv / fc
    return r - np.log(r) - 1

## Losses

Common sample: the (ETF, month) rows where all models have matching target dates and realized values.
The current selected models share forecast origins from February 2008 and target months from March 2008.

In [11]:
d = df[df.groupby(["etf", "date"])["model"].transform("nunique") == len(models)].copy()
d = d.sort_values(["date", "etf", "model"]).reset_index(drop=True)
if d.empty:
    raise ValueError("No common forecast sample across the models.")
values = ["actual_logRV", "actual_RV", "fc_log", "fc_level"]
if not np.isfinite(d[values].to_numpy()).all():
    raise ValueError("The common sample contains non-finite forecasts or outcomes.")
if (d[["actual_RV", "fc_level"]] <= 0).any().any():
    raise ValueError("QLIKE requires positive realized and forecast variances.")
keys = ["etf", "date"]
if (d.groupby(keys)["target_date"].nunique() != 1).any():
    raise ValueError("Target months differ across models.")
for col in ["actual_logRV", "actual_RV"]:
    wide = d.pivot(index=keys, columns="model", values=col)
    if not np.allclose(wide.to_numpy(), wide[[BENCHMARK]].to_numpy(), rtol=1e-10, atol=1e-12):
        raise ValueError(f"Realized outcomes differ across models: {col}")
# Keep the pooled monthly average equally weighted across the five ETFs.
if (d.groupby("date")["etf"].nunique() != len(TICKERS)).any():
    raise ValueError("The common sample must contain all five ETFs each month.")

d["MSE_log"] = (d["actual_logRV"] - d["fc_log"]) ** 2
d["MSE_level"] = (d["actual_RV"] - d["fc_level"]) ** 2
d["QLIKE"] = qlike(d["actual_RV"], d["fc_level"])
print(f"Forecast origins: {d['date'].min():%Y-%m} to {d['date'].max():%Y-%m}")
print(f"Target months: {d['target_date'].min():%Y-%m} to {d['target_date'].max():%Y-%m}")
print(f"{d['date'].nunique()} months × {len(TICKERS)} ETFs × {len(models)} models = {len(d)} rows")

LOSSES = ["QLIKE", "MSE_log", "MSE_level"]
per_etf = d.groupby(["model", "etf"])[LOSSES].mean().reset_index()
pooled = d.groupby("model")[LOSSES].mean().reset_index().assign(etf="pooled")
losses = pd.concat([per_etf, pooled], ignore_index=True).rename(columns={"etf": "ETF"})
losses.to_csv(RESULTS_DIR / "e6_losses.csv", index=False)
losses[losses["ETF"] == "pooled"].set_index("model").loc[models, LOSSES].round(4)

Forecast origins: 2008-02 to 2025-11
Target months: 2008-03 to 2025-12
214 months × 5 ETFs × 11 models = 11770 rows


,QLIKE,MSE_log,MSE_level
model,,,
hist_rw,0.4063,0.6030,0.0001
garch,0.2635,0.4736,0.0001
garch_midas_baa10y,0.2682,0.4826,0.0001
en_no_macro,0.3279,0.5232,0.0001
en_no_vix,0.3938,0.5571,0.0002
en_vix_only,0.2953,0.4783,0.0013
en_full_macro,0.3366,0.4912,0.0002
xgb_no_macro,0.3854,0.5905,0.0001
xgb_no_vix,0.4032,0.5507,0.0002


## Tests

In [12]:
loss_idx = losses.set_index(["model", "ETF"])
rows = []
cw_components = []
for pair_id, a, b, _ in pairs:
    A = d[d["model"] == a].set_index(["date", "etf"]).sort_index()
    B = d[d["model"] == b].set_index(["date", "etf"]).sort_index()
    dm_diff = A["QLIKE"] - B["QLIKE"]   # negative favours a
    mse_diff = A["MSE_log"] - B["MSE_log"]
    comparisons = [("QLIKE", "DM", dm_diff), ("MSE_log", "DM", mse_diff)]
    if nested[a]:
        # Clark-West adjusted differential; positive favours a.
        adjustment = (B["fc_log"] - A["fc_log"]) ** 2
        cw = -mse_diff + adjustment
        comparisons.append(("MSE_log", "Clark-West", cw))
        # Reuse these components in the diagnostic table.
        cw_components.append({
            "pair": pair_id, "a": a,
            "mean_raw_gain": (-mse_diff).groupby("date").mean().mean(),
            "mean_adjustment_term": adjustment.groupby("date").mean().mean(),
            "mean_CW_differential": cw.groupby("date").mean().mean(),
        })

    for etf in ["pooled"] + TICKERS:
        for loss, test, diff in comparisons:
            series = diff.groupby("date").mean() if etf == "pooled" else diff.xs(etf, level="etf")
            series = series.sort_index()
            if len(series) < 2:
                raise ValueError(f"Insufficient observations for {pair_id}, {etf}.")
            t = -hac_t(series) if test == "DM" else hac_t(series)
            rows.append({
                "pair": pair_id, "a": a, "b": b, "ETF": etf, "loss": loss, "test": test,
                "loss_ratio": loss_idx.loc[(a, etf), loss] / loss_idx.loc[(b, etf), loss],
                "gain_t": t, "p_value": float(norm.sf(t)),
            })

tests = pd.DataFrame(rows)
tests["pct_change"] = (tests["loss_ratio"] - 1) * 100  # negative: a has the lower raw loss
tests.to_csv(RESULTS_DIR / "e6_evaluation.csv", index=False)
tests[tests["ETF"] == "pooled"].drop(columns="ETF").round(4).reset_index(drop=True)

,pair,a,b,loss,test,loss_ratio,gain_t,p_value,pct_change
0,E6.1,garch,hist_rw,QLIKE,DM,0.6487,4.1034,0.0000,-35.1327
1,E6.1,garch,hist_rw,MSE_log,DM,0.7854,3.8433,0.0001,-21.4617
2,E6.2,garch_midas_baa10y,hist_rw,QLIKE,DM,0.6602,4.0567,0.0000,-33.9797
3,E6.2,garch_midas_baa10y,hist_rw,MSE_log,DM,0.8004,3.5340,0.0002,-19.9610
4,E6.3,en_no_macro,hist_rw,QLIKE,DM,0.8070,3.1702,0.0008,-19.2990
5,E6.3,en_no_macro,hist_rw,MSE_log,DM,0.8677,3.8839,0.0001,-13.2328
6,E6.3,en_no_macro,hist_rw,MSE_log,Clark-West,0.8677,6.0559,0.0000,-13.2328
7,E6.4,en_no_vix,hist_rw,QLIKE,DM,0.9692,0.2248,0.4111,-3.0803
8,E6.4,en_no_vix,hist_rw,MSE_log,DM,0.9240,1.0610,0.1443,-7.6010
9,E6.4,en_no_vix,hist_rw,MSE_log,Clark-West,0.9240,6.8569,0.0000,-7.6010


Clark-West components for the nested (Elastic Net) pairs, pooled across ETFs: the raw log-MSE gain
of `a` over `hist_rw`, the nesting adjustment, and their sum.

In [13]:
pd.DataFrame(cw_components).set_index("pair").round(4)

,a,mean_raw_gain,mean_adjustment_term,mean_CW_differential
pair,,,,
E6.3,en_no_macro,0.0798,0.0562,0.1359
E6.4,en_no_vix,0.0458,0.2112,0.2570
E6.5,en_vix_only,0.1247,0.1585,0.2832
E6.6,en_full_macro,0.1117,0.2237,0.3354


## By sector tests (exploratory)

Each cell is `loss_ratio (p-value)`; a ratio below 1 means the model is better than `hist_rw`.
Five sectors times ten models with no multiple-testing correction, so a single sector's p-value
is weak evidence; the pooled column is the formal result.

In [14]:
from IPython.display import display

SECTORS = {
    "XLB": "Materials", "XLE": "Energy", "XLF": "Financials",
    "XLI": "Industrials", "XLY": "Cons. Disc.",
}


def fmt_p(p):
    return "<0.001" if p < 0.001 else f"{p:.3f}"


def by_sector(loss, test):
    t = tests[tests["loss"].eq(loss) & tests["test"].eq(test)].copy()
    t["cell"] = [f"{ratio:.3f} ({fmt_p(p)})" for ratio, p in zip(t["loss_ratio"], t["p_value"])]
    t["row"] = t["pair"] + ": " + t["a"] + " vs " + t["b"]
    table = t.pivot(index="row", columns="ETF", values="cell").reindex(columns=TICKERS + ["pooled"])
    table = table.loc[t["row"].drop_duplicates()]
    table.columns = [f"{ticker} {SECTORS[ticker]}" for ticker in TICKERS] + ["Pooled"]
    return table.rename_axis(None)


for loss, test in [("MSE_log", "DM"), ("MSE_log", "Clark-West"), ("QLIKE", "DM")]:
    print(f"{loss}: ratio (p-value), {test}")
    display(by_sector(loss, test))

MSE_log: ratio (p-value), DM


,XLB Materials,XLE Energy,XLF Financials,XLI Industrials,XLY Cons. Disc.,Pooled
E6.1: garch vs hist_rw,0.804 (0.001),0.826 (<0.001),0.762 (<0.001),0.726 (<0.001),0.827 (0.005),0.785 (<0.001)
E6.2: garch_midas_baa10y vs hist_rw,0.815 (0.003),0.848 (0.006),0.767 (<0.001),0.751 (<0.001),0.842 (0.008),0.800 (<0.001)
E6.3: en_no_macro vs hist_rw,0.854 (<0.001),0.884 (0.006),0.880 (<0.001),0.841 (<0.001),0.878 (<0.001),0.868 (<0.001)
E6.4: en_no_vix vs hist_rw,0.859 (0.015),1.076 (0.725),0.843 (0.006),0.947 (0.346),0.932 (0.200),0.924 (0.144)
E6.5: en_vix_only vs hist_rw,0.757 (<0.001),0.876 (0.021),0.746 (<0.001),0.729 (0.004),0.886 (0.093),0.793 (<0.001)
E6.6: en_full_macro vs hist_rw,0.776 (0.001),1.057 (0.677),0.763 (<0.001),0.696 (<0.001),0.833 (0.008),0.815 (0.002)
E6.7: xgb_no_macro vs hist_rw,0.981 (0.396),1.160 (0.906),0.979 (0.378),0.880 (0.038),0.927 (0.141),0.979 (0.370)
E6.8: xgb_no_vix vs hist_rw,0.914 (0.124),1.127 (0.834),0.869 (0.067),0.823 (0.007),0.880 (0.033),0.913 (0.108)
E6.9: xgb_vix_only vs hist_rw,0.838 (0.014),1.104 (0.815),0.873 (0.063),0.714 (<0.001),0.915 (0.141),0.880 (0.035)
E6.10: xgb_full_macro vs hist_rw,0.804 (0.006),1.058 (0.669),0.803 (0.011),0.708 (<0.001),0.801 (0.002),0.826 (0.006)


MSE_log: ratio (p-value), Clark-West


,XLB Materials,XLE Energy,XLF Financials,XLI Industrials,XLY Cons. Disc.,Pooled
E6.3: en_no_macro vs hist_rw,0.854 (<0.001),0.884 (<0.001),0.880 (<0.001),0.841 (<0.001),0.878 (<0.001),0.868 (<0.001)
E6.4: en_no_vix vs hist_rw,0.859 (<0.001),1.076 (<0.001),0.843 (<0.001),0.947 (<0.001),0.932 (<0.001),0.924 (<0.001)
E6.5: en_vix_only vs hist_rw,0.757 (<0.001),0.876 (<0.001),0.746 (<0.001),0.729 (<0.001),0.886 (<0.001),0.793 (<0.001)
E6.6: en_full_macro vs hist_rw,0.776 (<0.001),1.057 (<0.001),0.763 (<0.001),0.696 (<0.001),0.833 (<0.001),0.815 (<0.001)


QLIKE: ratio (p-value), DM


,XLB Materials,XLE Energy,XLF Financials,XLI Industrials,XLY Cons. Disc.,Pooled
E6.1: garch vs hist_rw,0.672 (<0.001),0.733 (<0.001),0.581 (<0.001),0.621 (<0.001),0.674 (<0.001),0.649 (<0.001)
E6.2: garch_midas_baa10y vs hist_rw,0.664 (<0.001),0.759 (0.003),0.592 (<0.001),0.637 (<0.001),0.686 (<0.001),0.660 (<0.001)
E6.3: en_no_macro vs hist_rw,0.825 (0.009),0.852 (0.026),0.769 (0.003),0.815 (0.003),0.792 (<0.001),0.807 (<0.001)
E6.4: en_no_vix vs hist_rw,0.884 (0.176),1.192 (0.793),0.861 (0.171),1.049 (0.597),0.907 (0.188),0.969 (0.411)
E6.5: en_vix_only vs hist_rw,0.710 (<0.001),0.839 (0.044),0.667 (0.003),0.687 (<0.001),0.766 (0.004),0.727 (<0.001)
E6.6: en_full_macro vs hist_rw,0.790 (0.032),1.130 (0.722),0.763 (0.017),0.708 (<0.001),0.812 (0.021),0.829 (0.035)
E6.7: xgb_no_macro vs hist_rw,0.997 (0.492),1.179 (0.776),0.900 (0.194),0.866 (0.088),0.850 (0.049),0.949 (0.317)
E6.8: xgb_no_vix vs hist_rw,0.953 (0.345),1.294 (0.858),0.962 (0.419),0.887 (0.190),0.912 (0.150),0.992 (0.477)
E6.9: xgb_vix_only vs hist_rw,0.854 (0.126),1.071 (0.667),0.859 (0.144),0.775 (0.015),0.841 (0.064),0.874 (0.111)
E6.10: xgb_full_macro vs hist_rw,0.849 (0.117),1.231 (0.785),0.854 (0.155),0.757 (0.007),0.777 (0.007),0.884 (0.165)


## By sector percentage changes

Percentage change in mean loss for each model relative to `hist_rw`:
`100 × (loss_ratio - 1)`. Negative values mean lower loss than the benchmark;
positive values mean higher loss. Log MSE and QLIKE use the same common sample
as the tests above. The pooled column includes all five sectors.

In [15]:
for loss in ["MSE_log", "QLIKE"]:
    pct = (
        tests.loc[tests["loss"].eq(loss) & tests["test"].eq("DM")]
        .assign(row=lambda t: t["pair"] + ": " + t["a"])
        .pivot(index="row", columns="ETF", values="pct_change")
        .reindex(columns=TICKERS + ["pooled"])
    )
    pct = pct.loc[[f"{pair_id}: {a}" for pair_id, a, _, _ in pairs]]
    pct.columns = [f"{ticker} {SECTORS[ticker]}" for ticker in TICKERS] + ["Pooled"]
    print(f"{'Log MSE' if loss == 'MSE_log' else loss}: % change relative to {BENCHMARK}")
    display(pct.rename_axis(None).map(lambda value: f"{value:+.2f}%"))

Log MSE: % change relative to hist_rw


,XLB Materials,XLE Energy,XLF Financials,XLI Industrials,XLY Cons. Disc.,Pooled
E6.1: garch,-19.63%,-17.36%,-23.77%,-27.37%,-17.33%,-21.46%
E6.2: garch_midas_baa10y,-18.47%,-15.24%,-23.31%,-24.87%,-15.78%,-19.96%
E6.3: en_no_macro,-14.57%,-11.56%,-12.01%,-15.86%,-12.19%,-13.23%
E6.4: en_no_vix,-14.13%,+7.60%,-15.71%,-5.34%,-6.80%,-7.60%
E6.5: en_vix_only,-24.29%,-12.42%,-25.41%,-27.11%,-11.38%,-20.68%
E6.6: en_full_macro,-22.38%,+5.70%,-23.69%,-30.38%,-16.70%,-18.53%
E6.7: xgb_no_macro,-1.95%,+15.99%,-2.07%,-12.02%,-7.32%,-2.06%
E6.8: xgb_no_vix,-8.58%,+12.70%,-13.11%,-17.72%,-12.01%,-8.67%
E6.9: xgb_vix_only,-16.20%,+10.36%,-12.71%,-28.65%,-8.53%,-11.95%
E6.10: xgb_full_macro,-19.62%,+5.76%,-19.66%,-29.17%,-19.88%,-17.37%


QLIKE: % change relative to hist_rw


,XLB Materials,XLE Energy,XLF Financials,XLI Industrials,XLY Cons. Disc.,Pooled
E6.1: garch,-32.78%,-26.67%,-41.89%,-37.92%,-32.64%,-35.13%
E6.2: garch_midas_baa10y,-33.62%,-24.14%,-40.76%,-36.27%,-31.38%,-33.98%
E6.3: en_no_macro,-17.49%,-14.77%,-23.09%,-18.49%,-20.80%,-19.30%
E6.4: en_no_vix,-11.62%,+19.18%,-13.89%,+4.92%,-9.30%,-3.08%
E6.5: en_vix_only,-28.99%,-16.07%,-33.25%,-31.34%,-23.39%,-27.33%
E6.6: en_full_macro,-20.96%,+12.97%,-23.66%,-29.22%,-18.77%,-17.15%
E6.7: xgb_no_macro,-0.28%,+17.91%,-10.01%,-13.43%,-15.03%,-5.15%
E6.8: xgb_no_vix,-4.69%,+29.43%,-3.85%,-11.32%,-8.83%,-0.77%
E6.9: xgb_vix_only,-14.60%,+7.15%,-14.12%,-22.50%,-15.90%,-12.63%
E6.10: xgb_full_macro,-15.13%,+23.13%,-14.56%,-24.28%,-22.27%,-11.65%


## Diagnostics: largest level errors

For each model, the share of its level MSE that comes from its five largest ETF-month errors,
then the ten largest level errors across all models.

In [16]:
level_errors = d.copy()
level_errors["level_ratio"] = level_errors.fc_level / level_errors.actual_RV
level_errors["level_error_share"] = (
    level_errors.MSE_level / level_errors.groupby("model").MSE_level.transform("sum")
)
top5_share = (level_errors.sort_values("MSE_level", ascending=False)
              .groupby("model").head(5).groupby("model")["level_error_share"].sum())
print("Share of level MSE from each model's five largest errors:")
display(top5_share.loc[models].round(3).to_frame())

error_cols = ["model", "etf", "date", "target_date", "actual_RV", "fc_log", "sigma2", "fc_level",
              "level_ratio", "MSE_log", "MSE_level", "QLIKE", "level_error_share"]
level_errors.nlargest(10, "MSE_level")[error_cols]

Share of level MSE from each model's five largest errors:


,level_error_share
model,
hist_rw,0.435
garch,0.504
garch_midas_baa10y,0.493
en_no_macro,0.524
en_no_vix,0.604
en_vix_only,0.917
en_full_macro,0.568
xgb_no_macro,0.433
xgb_no_vix,0.457


,model,etf,date,target_date,actual_RV,fc_log,sigma2,fc_level,level_ratio,MSE_log,MSE_level,QLIKE,level_error_share
476,en_vix_only,XLI,2008-10-31,2008-11-30,0.024007,-0.351125,0.248288,0.796936,33.196234,11.412859,0.597419,2.532560,0.432892
487,en_vix_only,XLY,2008-10-31,2008-11-30,0.045574,-0.484902,0.289039,0.711498,15.611903,6.778285,0.443455,1.812087,0.321329
443,en_vix_only,XLB,2008-10-31,2008-11-30,0.035719,-1.149422,0.287944,0.365880,10.243234,4.763941,0.109006,1.424243,0.078986
465,en_vix_only,XLF,2008-10-31,2008-11-30,0.087358,-1.210195,0.549502,0.392412,4.491983,1.506862,0.093058,0.724913,0.067430
629,en_no_vix,XLF,2009-01-31,2009-02-28,0.054068,-1.844775,0.806471,0.236563,4.375321,1.150781,0.033305,0.704535,0.200863
484,en_full_macro,XLY,2008-10-31,2008-11-30,0.045574,-1.896338,0.730596,0.216311,4.746352,1.421051,0.029151,0.768064,0.152959
462,en_full_macro,XLF,2008-10-31,2008-11-30,0.087358,-1.834815,0.886359,0.248668,2.846536,0.363516,0.026021,0.397407,0.136535
7933,en_no_vix,XLE,2020-02-29,2020-03-31,0.162331,-5.189309,0.622527,0.007612,0.046891,11.364948,0.023938,17.266143,0.144373
7940,xgb_no_vix,XLE,2020-02-29,2020-03-31,0.162331,-5.176799,0.682218,0.007941,0.048920,11.280760,0.023836,16.424087,0.147115
7938,xgb_full_macro,XLE,2020-02-29,2020-03-31,0.162331,-5.021976,0.714636,0.009422,0.058045,10.264728,0.023381,13.381584,0.144420
